# Risco alimentar e vulnerabilidade social no Brasil — versão 2

**Pergunta:** em quais municípios brasileiros os sinais de vulnerabilidade
social estrutural, risco de insegurança alimentar e vulnerabilidade nutricional
infantil aparecem simultaneamente?

Este estudo ecológico municipal utiliza IVS, IDHM, CadInsan e SISVAN em
**três dimensões**. IVS e IDHM compõem juntos a dimensão social; CadInsan,
a alimentar; DAI, a nutricional. DPI é complementar. A classificação
identifica convergência, **não um ranking, diagnóstico municipal de fome
ou relação causal**. Cada município é uma unidade; não se combinam
pessoas e famílias em um único denominador.

## 1. Fontes de dados e procedência

| Entrada | Instituição / fonte original | Arquivo utilizado e obtenção | Referência / população |
|---|---|---|---|
| IVS e IDHM | [Atlas IVS/Ipea](https://repositorio.ipea.gov.br/handle/11058/4381); [Atlas Brasil/PNUD, Ipea e FJP](https://www.undp.org/pt/brazil/desenvolvimento-humano/atlas-do-desenvolvimento-humano-no-brasil) | [CSV de Cozinhas Solidárias](https://raw.githubusercontent.com/TriangulosTecnologia/cozsolidarias/refs/heads/main/src/data-source-static/data/atlasivs_municipios_2010.csv), 5.565 municípios | 2010; índices municipais |
| CadInsan | [MDS, relatório 2025](https://www.gov.br/mds/pt-br/Sisan/vigilancia-do-sisan/CADINSAN2025.pdf) | [CSV de Cozinhas Solidárias](https://raw.githubusercontent.com/TriangulosTecnologia/cozsolidarias/refs/heads/main/src/data-source-static/data/CADINSAN_2025_dados_municipais.csv), 5.570 municípios | Janeiro/2025; famílias do universo analisado |
| SISVAN — Altura X Idade | [Ministério da Saúde, relatórios públicos](https://sisaps.saude.gov.br/sisvan/relatoriopublico/) | CSV municipal da coleta direta de 4/10/2026, consolidada à tarde; 5.571 municípios | 2025; crianças de 0 a menos de 5 anos acompanhadas pelo sistema |
| SISVAN — Peso X Idade | Ministério da Saúde, mesmos relatórios públicos | CSV municipal da coleta já disponível, consolidada em 4/10/2026 pela manhã; 5.571 municípios | 2025; mesmo recorte etário e 27 UFs |

As **duas entradas sociais** foram obtidas do repositório de
[Cozinhas Solidárias](https://github.com/TriangulosTecnologia/cozsolidarias),
não extraídas diretamente dos portais oficiais nesta pesquisa. Seus hashes
são conferidos contra os respectivos registros do catálogo fornecido com elas.
Os dois CSVs nutricionais e seus metadados são reutilizados sem modificar
os arquivos do projeto. DAI e DPI são calculados aqui; a antiga base combinada
com indicadores derivados não é utilizada.

**Por que não há uma entrada adicional do CadÚnico?** O CadInsan já utiliza
famílias inscritas nesse cadastro. Para esta estratégia, acrescentar apenas
o número absoluto de pessoas cadastradas repetiria parte do contexto conceitual
e refletiria fortemente o tamanho populacional. Essa é uma decisão da análise,
não uma afirmação de que todas as variáveis do CadÚnico sejam redundantes.
O denominador de famílias do próprio CSV CadInsan continua necessário.

A malha simplificada é da [API v4 do IBGE](https://servicodados.ibge.gov.br/api/docs/malhas?versao=4),
incorporada com hash e metadados. A API usada não identifica seu ano: os mapas
são apoio ilustrativo, não prova de harmonização dos limites entre 2010 e 2025.

**Como reproduzir:** abrir este arquivo no Google Colab e selecionar
**Ambiente de execução → Executar tudo**. Entradas, código e malha estão
incorporados; não é necessário ter os demais arquivos do projeto ou credenciais.
Apenas a instalação inicial de bibliotecas pode precisar de internet.

## 2. Indicadores, dimensões e escolhas metodológicas

**Social:** IVS descreve infraestrutura urbana, capital humano e renda/trabalho;
quanto maior, maior a vulnerabilidade. É publicado como média dos três
subíndices. IDHM sintetiza longevidade, educação e renda por média geométrica
de seus índices; quanto menor, menor o desenvolvimento. Ambos são lidos da
fonte, não reconstruídos a partir de microdados. A dimensão exige
**IVS ≥ 0,401 E IDHM < 0,600**: faixas alta/muito alta do IVS/Ipea e
baixo/muito baixo desenvolvimento adotado aqui. IDHM igual a 0,600 não entra.
São aspectos relacionados; a correlação de Spearman será medida, e não
se contam os dois como dimensões independentes.

**Alimentar:** CadInsan estima risco de insegurança alimentar grave entre
famílias do universo analisado, não mede diretamente fome em todas as famílias
residentes. Utiliza-se **CadInsan ≥ P75 nacional**. P75 é uma escolha
distributiva/exploratória, **não um corte oficial**: identifica o quartil superior
municipal, aproximadamente 25% dos municípios válidos, com empates incluídos.
Cada município recebe o mesmo peso; não são 25% da população. O quantil
é calculado antes de filtrar pelos critérios sociais ou nutricionais.

`com_PBF` considera o efeito do Bolsa Família na renda; `sem_PBF` é o
contrafactual que desconsidera esse efeito. Não são grupos de beneficiários
versus não beneficiários. O CSV é referente a famílias com cadastro atualizado
nos últimos 12 meses, em janeiro/2025.

**Nutricional:** utiliza-se **DAI ≥ 6,7% com pelo menos 20 avaliações de altura**.
A referência vem dos critérios de inclusão do
[Mapa InSAN 2017–2022, seção de mapeamento, tabela 2](https://www.gov.br/mds/pt-br/Sisan/monitoramento-da-san/MapaInSAN_20172022.pdf).
Seus limiares de DAI (6,7%) e DPI (1,8%) retomam a PNDS 2006; a exclusão
de municípios com menos de 20 acompanhamentos é um precedente operacional.
Não são novos limites clínicos da OMS nem garantia de precisão amostral.

**Adaptação, não reprodução do Mapa InSAN:** o Mapa se refere a crianças
beneficiárias do PBF e utiliza também agrupamento estatístico. Aqui os relatórios
não filtram exclusivamente esse público, DAI é o critério principal e não há
clusterização. A comparação DAI+DPI exige os mínimos separadamente em cada
relatório, sem afirmar que seus denominadores representam as mesmas crianças.
DPI permanece complementar e não interfere na seleção principal.

In [1]:
QUANTIL_CADINSAN = 0.75
CORTE_DAI = 6.7  # porcentagem; referência metodológica do Mapa InSAN
CORTE_DPI = 1.8  # porcentagem; indicador complementar
MINIMO_AVALIADOS = 20
CENARIO_CADINSAN = "com_PBF"
QUANTIS_SENSIBILIDADE = [0.70, 0.75, 0.80, 0.90]
MINIMOS_SENSIBILIDADE = [20, 50, 100]
CORTES_DAI_SENSIBILIDADE = [6.7, 10.0, "P75"]
CENARIOS_SENSIBILIDADE = ["com_PBF", "sem_PBF"]
GERAR_MAPAS = True
BAIXAR_RESULTADOS_NO_COLAB = False

### Preparação do ambiente

As células técnicas recolhidas preparam bibliotecas e recuperam as entradas.
Os resultados ficam separados dos arquivos da fonte. Contagens, índices e
percentuais são apresentados no padrão brasileiro; `—` indica ausência.
A formatação de tabelas não altera valores, cálculos ou CSVs exportados.

In [2]:
import importlib.util
import subprocess
import sys
from pathlib import Path

packages = {"pandas": "pandas>=2.2,<4", "numpy": "numpy>=1.26,<3",
            "matplotlib": "matplotlib>=3.8,<4", "scipy": "scipy>=1.11,<2",
            "jinja2": "jinja2>=3.1,<4"}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import hashlib
import io
import json
import zipfile
import base64
import platform
import shutil
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
from IPython.display import display, Markdown

PASTA_EXECUCAO = Path.cwd() / "resultados_sobreposicao_v2"
PASTA_DADOS = PASTA_EXECUCAO / "entradas"
PASTA_SAIDA = PASTA_EXECUCAO / "resultados"
PASTA_FIGURAS = PASTA_SAIDA / "figuras"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)

"""Formatação brasileira de tabelas, sem transformar os dados analíticos."""

import pandas as pd
from pandas.api.types import is_bool_dtype, is_integer_dtype, is_numeric_dtype


COLUNAS_CONTAGEM = {
    "municipios", "registros_origem", "municipios_na_uniao",
    "correspondencias_sisvan", "sisvan_sem_fonte", "fonte_sem_sisvan",
    "numerador", "denominador", "municipios_referencia", "municipios_no_criterio",
    "percentil", "elegiveis", "prioritarios", "coincidentes_principal",
    "minimo_avaliados", "cenarios_elegiveis", "cenarios_selecionado", "cenarios_testados",
    "familias_cadinsan", "convergencia_total", "entram_vs_principal", "saem_vs_principal",
    "municipios_dim_alimentar", "municipios_dim_nutricional", "dai_isolado", "dai_e_dpi",
    "convergencia_dai", "convergencia_dai_dpi",
}


def numero_br(valor, casas=3):
    if pd.isna(valor):
        return "—"
    return f"{valor:,.{casas}f}".translate(str.maketrans({",": ".", ".": ","}))


def contagem_br(valor):
    return numero_br(valor, 0)


def percentual_br(valor):
    return "—" if pd.isna(valor) else numero_br(valor, 2) + "%"


def quantil_br(valor):
    return numero_br(valor, 2)


def tabela_br(tabela):
    """Retorna apenas um Styler; mantém valores, tipos, índices e colunas.

    Percentuais já estão na escala 0–100. Não multiplica razões por 100,
    não converte códigos em números e não muda contagens na base exportada.
    """
    formatos = {}
    for coluna in tabela.columns:
        nome = str(coluna).lower()
        if is_bool_dtype(tabela[coluna]):
            continue
        if nome in COLUNAS_CONTAGEM or nome.endswith("_n") or nome.startswith(
            ("avaliados_", "pessoas ", "famílias ", "avaliações ")
        ):
            formatos[coluna] = contagem_br
        elif "(%)" in nome or "_pct" in nome or nome.startswith("pct_") or nome == "percentual_agregado":
            formatos[coluna] = percentual_br
        elif nome == "quantil":
            formatos[coluna] = quantil_br
        elif is_numeric_dtype(tabela[coluna]):
            formatos[coluna] = contagem_br if is_integer_dtype(tabela[coluna]) else numero_br
    visualizacao = tabela.style.format(formatos, na_rep="—", escape="html")
    # O corte combina índices (0–1) e percentuais (0–100) na mesma coluna.
    if {"indicador", "corte"}.issubset(tabela.columns):
        indices = tabela.index[tabela["indicador"].isin(["ivs", "idhm"])]
        percentuais = tabela.index[tabela["indicador"].isin(["cadinsan", "dai", "dpi"])]
        if len(indices):
            visualizacao.format(numero_br, subset=(indices, ["corte"]), na_rep="—")
        if len(percentuais):
            visualizacao.format(percentual_br, subset=(percentuais, ["corte"]), na_rep="—")
    return visualizacao


CODIGO_ANALISE_SHA256 = "688983c8d1ded9557d8cc96b5123c2776650af2b5b9fa480df6c06d51f55a27a"

In [3]:
# @title Bases e malha incorporadas — executar sem editar
PACOTE_BASE64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACOTE_BASE64))) as pacote:
    for membro in pacote.infolist():
        destino = (PASTA_DADOS / membro.filename).resolve()
        if not destino.is_relative_to(PASTA_DADOS.resolve()):
            raise ValueError("Caminho inválido no pacote de dados")
    pacote.extractall(PASTA_DADOS)
print("Bases recuperadas para análise.")

Bases recuperadas para análise.


In [4]:
# @title Funções de análise — executar sem editar
"""Três dimensões municipais; mantém leitores e cartografia da primeira versão."""

from __future__ import annotations

import hashlib
import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

from decimal import Decimal, InvalidOperation

UF_REGIONS = {
    "11": ("RO", "Norte"), "12": ("AC", "Norte"),
    "13": ("AM", "Norte"), "14": ("RR", "Norte"),
    "15": ("PA", "Norte"), "16": ("AP", "Norte"),
    "17": ("TO", "Norte"), "21": ("MA", "Nordeste"),
    "22": ("PI", "Nordeste"), "23": ("CE", "Nordeste"),
    "24": ("RN", "Nordeste"), "25": ("PB", "Nordeste"),
    "26": ("PE", "Nordeste"), "27": ("AL", "Nordeste"),
    "28": ("SE", "Nordeste"), "29": ("BA", "Nordeste"),
    "31": ("MG", "Sudeste"), "32": ("ES", "Sudeste"),
    "33": ("RJ", "Sudeste"), "35": ("SP", "Sudeste"),
    "41": ("PR", "Sul"), "42": ("SC", "Sul"), "43": ("RS", "Sul"),
    "50": ("MS", "Centro-Oeste"), "51": ("MT", "Centro-Oeste"),
    "52": ("GO", "Centro-Oeste"), "53": ("DF", "Centro-Oeste"),
}

def numeric(values, label):
    """Lê ponto/vírgula decimal e %, sem converter ausência em zero."""
    clean = values.astype("string").str.strip()
    clean = clean.str.replace("%", "", regex=False).str.replace(",", ".", regex=False)
    present = clean.notna() & clean.ne("")
    result = pd.to_numeric(clean.mask(~present), errors="coerce").astype(float)
    invalid = present & result.isna()
    if invalid.any():
        raise ValueError(f"Valores não numéricos em {label}: {values[invalid].head().tolist()}")
    if not np.isfinite(result.dropna()).all():
        raise ValueError(f"Valores infinitos em {label}")
    return result

def codes(values, length, source):
    result = values.astype("string").str.strip()
    if not result.str.fullmatch(rf"\d{{{length}}}").fillna(False).all():
        raise ValueError(f"{source}: códigos devem ter {length} dígitos.")
    if result.duplicated().any():
        raise ValueError(f"{source}: código municipal duplicado.")
    return result

def check_range(values, low, high, label):
    if not values.dropna().between(low, high).all():
        raise ValueError(f"{label}: valores fora de [{low}, {high}].")

def interpret_sisvan_counts(values, percentages, code):
    """Interpreta artefatos numéricos sem modificar as células de entrada.

    Mantém inteiros coerentes sem criar uma alternativa uniforme ×1000.
    Decimais do XLSX são interpretados como milhares. Apenas quando a soma
    ou os percentuais não conferem, tenta conciliar células truncadas pelo
    exportador. Exige uma solução única, soma exata e percentuais a 0,011 pp.
    """
    original_values, initial, integral = [], [], []
    for raw in values:
        text = str(raw).strip()
        try:
            value = Decimal(text)
        except InvalidOperation as exc:
            raise ValueError(f"{code}: contagem SISVAN não numérica: {raw!r}") from exc
        if not value.is_finite() or value < 0:
            raise ValueError(f"{code}: contagem SISVAN inválida: {raw!r}")
        original_values.append(value)
        if value == value.to_integral_value():
            initial.append(int(value))
            integral.append(True)
        else:
            scaled = value * 1000
            if scaled != scaled.to_integral_value():
                raise ValueError(f"{code}: escala decimal SISVAN irresolvível: {raw!r}")
            initial.append(int(scaled))
            integral.append(False)
    pcts = []
    for raw in percentages:
        text = str(raw).strip()
        pct = 0.0 if text in {"-", "–", "—"} else float(text.rstrip("%").replace(",", "."))
        if not np.isfinite(pct) or not 0 <= pct <= 100:
            raise ValueError(f"{code}: percentual SISVAN inválido: {raw!r}")
        pcts.append(pct)

    def consistent(counts, total):
        if sum(counts) != total:
            return False
        if total == 0:
            return not any(pcts)
        return all(abs(n / total * 100 - pct) <= 0.011 for n, pct in zip(counts, pcts))

    # Mantém as contagens inteiras quando soma e percentuais conferem.
    if consistent(initial[:-1], initial[-1]):
        changed = any(value != n for value, n in zip(original_values, initial))
        return tuple(initial), changed, 1

    # Exportações podem perder zeros finais (ex.: 1.000 vira 1). Essa leitura
    # só é tentada para linhas inconsistentes, não para criar outra população.
    candidates = [[n, n * 1000] if is_int and n else [n]
                  for n, is_int in zip(initial, integral)]
    solutions = []
    for counts in product(*candidates[:-1]):
        total = sum(counts)
        if total not in candidates[-1]:
            continue
        if not consistent(counts, total):
            continue
        solutions.append((*counts, total))
    if not solutions:
        raise ValueError(f"{code}: contagens não conciliam soma e percentuais oficiais.")
    if len(solutions) != 1:
        raise ValueError(f"{code}: contagens inconsistentes admitem mais de uma interpretação; conferir na fonte.")
    chosen = solutions[0]
    changed = any(value != n for value, n in zip(original_values, chosen))
    return chosen, changed, 1

def figure_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "figure.dpi": 110,
                         "savefig.dpi": 180})

def save_figure(fig, output, name):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"{name}.png", bbox_inches="tight")
    fig.savefig(output / f"{name}.svg", bbox_inches="tight")

def geometry_audit(geojson, base):
    features = geojson["features"]
    codes_7 = pd.Series([str(f["properties"]["codarea"]) for f in features], dtype="string")
    codes(codes_7, 7, "Malha IBGE")
    prefixes = codes_7.str[:6]
    if prefixes.duplicated().any():
        raise ValueError("Malha com prefixos municipais duplicados.")
    existing = dict(zip(prefixes, codes_7))
    known = base["codigo_ibge_7"].notna()
    for row in base.loc[known].itertuples():
        if row.codigo_ibge_6 in existing and existing[row.codigo_ibge_6] != row.codigo_ibge_7:
            raise ValueError("Conflito de código municipal entre malha e bases sociais.")
    return pd.DataFrame({"codigo_ibge_6": base["codigo_ibge_6"],
                         "codigo_ibge_7_malha": base["codigo_ibge_6"].map(existing),
                         "tem_geometria": base["codigo_ibge_6"].isin(existing)})

def map_figure(geojson, result, column, title, output, filename, categories=None):
    """Mapa ilustrativo em coordenadas geográficas; anéis interiores preservados."""
    from textwrap import fill
    import matplotlib.pyplot as plt
    from matplotlib.collections import PatchCollection
    from matplotlib.colors import Normalize
    from matplotlib.path import Path as MplPath
    from matplotlib.patches import Patch, PathPatch
    lookup = result.set_index("codigo_ibge_6")[column].to_dict()
    patches, values, colors = [], [], []
    for feature in geojson["features"]:
        geometry = feature["geometry"]
        polygons = ([geometry["coordinates"]] if geometry["type"] == "Polygon"
                    else geometry["coordinates"])
        value = lookup.get(str(feature["properties"]["codarea"])[:6], np.nan)
        for polygon in polygons:
            vertices, commands = [], []
            for ring in polygon:
                coordinates = np.asarray(ring, dtype=float)[:, :2]
                if len(coordinates) < 4:
                    continue
                vertices.extend(coordinates)
                commands.extend([MplPath.MOVETO] + [MplPath.LINETO] * (len(coordinates) - 2) + [MplPath.CLOSEPOLY])
            if vertices:
                patches.append(PathPatch(MplPath(vertices, commands)))
                if categories is not None:
                    colors.append(categories.get(value, "#b9b9b9"))
                else:
                    values.append(float(value) if pd.notna(value) else np.nan)
    fig, ax = plt.subplots(figsize=(9, 9))
    collection = PatchCollection(patches, linewidths=0.08, edgecolors="#ffffff")
    if categories is not None:
        collection.set_facecolors(colors)
        ax.legend(handles=[Patch(facecolor=color, label=fill(label, width=42)) for label, color in categories.items()],
                  loc="lower left", fontsize=8, frameon=False)
    else:
        arr = np.ma.masked_invalid(values)
        collection.set_array(arr)
        cmap = plt.get_cmap("YlOrRd_r" if column == "idhm" else "YlOrRd").copy()
        cmap.set_bad("#b9b9b9")
        collection.set_cmap(cmap)
        finite = np.asarray(values)[np.isfinite(values)]
        if finite.size:
            collection.set_norm(Normalize(vmin=float(finite.min()), vmax=float(finite.max())))
        fig.colorbar(collection, ax=ax, fraction=0.035, pad=0.015)
        ax.legend(handles=[Patch(facecolor="#b9b9b9", label="Ausente / não elegível")],
                  loc="lower left", frameon=False)
    ax.add_collection(collection)
    ax.set(xlim=(-74.5, -32), ylim=(-34.5, 6), title=title)
    ax.set_aspect("equal")
    ax.axis("off")
    fig.text(0.15, 0.075, "IBGE: malha simplificada de apoio; coordenadas geográficas.\n"
             "Mapa ilustrativo; não mede cobertura ou concentração espacial estatística.", fontsize=8)
    save_figure(fig, output, filename)
    return fig

DATA_FILES = {
    "ivs_idhm": "ivs_idhm/atlasivs_municipios_2010.csv",
    "cadinsan": "cadinsan/CADINSAN_2025_dados_municipais.csv",
    "sisvan_altura": "sisvan/sisvan_municipios_altura_por_idade_0_a_menor_5_anos_2025.csv",
    "sisvan_peso": "sisvan/sisvan_municipios_peso_por_idade_0_a_menor_5_anos_2025.csv",
}
CATALOGUE_DATASETS = {"ivs_idhm": "municipios_ivs", "cadinsan": "municipios_cadinsan"}
FIXED_CUTS = {"ivs": (0.401, ">="), "idhm": (0.600, "<")}
DIMENSIONS = ["dim_social", "dim_alimentar", "dim_nutricional"]
PRIORITY_LABELS = {
    0: "Sem sinais pelos critérios adotados",
    1: "Atenção — uma dimensão",
    2: "Prioridade alta — duas dimensões",
    3: "Prioridade muito alta — três dimensões",
}
OMS_LABELS = ["muito baixa", "baixa", "média", "alta", "muito alta"]
OMS_COLORS = dict(zip(OMS_LABELS, ["#eff3ff", "#bdd7e7", "#fdcc8a", "#fc8d59", "#b30000"]))
OMS_COLORS["dados insuficientes"] = "#b9b9b9"
DIMENSION_COLORS = {"0": "#e0eee7", "1": "#fdd49e", "2": "#fc8d59",
                    "3": "#9e1b32", "dados insuficientes": "#b9b9b9"}


def verify_hashes(directory):
    directory = Path(directory)
    records = []
    for line in (directory / "SHA256SUMS").read_text().splitlines():
        if not line.strip():
            continue
        expected, relative = line.split(maxsplit=1)
        relative = relative.lstrip("*")
        actual = hashlib.sha256((directory / relative).read_bytes()).hexdigest()
        if actual != expected:
            raise ValueError(f"Integridade inválida: {relative}")
        records.append({"arquivo": relative, "sha256": actual, "integridade": "OK"})
    if len(records) != len(DATA_FILES) or {r["arquivo"] for r in records} != set(DATA_FILES.values()):
        raise ValueError("O manifesto deve identificar exatamente as quatro entradas da v2.")
    return pd.DataFrame(records)


def validate_catalogue(catalogue_path, directory):
    raw = Path(catalogue_path).read_bytes()
    catalogue = json.loads(raw.decode("utf-8-sig"))
    records = []
    for source, dataset_id in CATALOGUE_DATASETS.items():
        dataset = catalogue["datasets"][dataset_id]
        relative = DATA_FILES[source]
        actual = hashlib.sha256((Path(directory) / relative).read_bytes()).hexdigest()
        if actual != dataset["stats"]["checkSum"].removeprefix("sha256:"):
            raise ValueError(f"Catálogo não corresponde a {relative}")
        temporal = dataset["temporal"]["extent"][0]
        records.append({"dataset_id": dataset_id, "arquivo": relative, "titulo": dataset["title"],
                        "inicio_referencia_catalogo": temporal[0], "fim_referencia_catalogo": temporal[1],
                        "fonte_original": dataset["source"]["url"], "sha256": actual,
                        "corresponde_ao_catalogo": True})
    return pd.DataFrame(records), {"catalogo_sha256": hashlib.sha256(raw).hexdigest(),
                                   "catalogo_id": catalogue["catalog"]["id"],
                                   "fontes": records, "recorte": "Somente fontes sociais usadas na v2"}


def sisvan_metadata(directory, source):
    path = Path(directory) / DATA_FILES[source]
    metadata = json.loads(path.with_suffix(".metadados.json").read_text())
    index = "altura_por_idade" if source == "sisvan_altura" else "peso_por_idade"
    expected = {"ano": 2025, "fase": "crianca", "indice": index,
                "faixa_etaria": "0_a_menor_5_anos", "nu_idade_inicio": "0", "nu_idade_fim": "5"}
    if any(metadata.get(key) != value for key, value in expected.items()):
        raise ValueError(f"{source}: recorte incompatível com menores de cinco anos, 2025")
    if metadata.get("indicadores_derivados", []) != []:
        raise ValueError("A entrada nutricional não deve conter DAI ou DPI pré-calculado")
    if "sha256_csv" in metadata and metadata["sha256_csv"] != hashlib.sha256(path.read_bytes()).hexdigest():
        raise ValueError(f"{source}: hash incompatível com os metadados")
    if set(metadata["ufs"]) != {uf for uf, _ in UF_REGIONS.values()}:
        raise ValueError(f"{source}: coleta nacional incompleta")
    return metadata


def read_nutrition(directory, source):
    metadata = sisvan_metadata(directory, source)
    raw = pd.read_csv(Path(directory) / DATA_FILES[source], dtype="string",
                      encoding="utf-8-sig", keep_default_na=False)
    if list(raw.columns) != metadata["colunas"] or len(raw) != metadata["linhas"]:
        raise ValueError(f"{source}: esquema ou linhas incompatíveis com os metadados")
    height = source == "sisvan_altura"
    parts = (["Altura Muito Baixa para a Idade", "Altura Baixa para a Idade",
              "Altura Adequada para a Idade"] if height else
             ["Peso Muito Baixo para a Idade", "Peso Baixo para a Idade",
              "Peso Adequado ou Eutrófico", "Peso Elevado para a Idade"])
    names = (["altura_muito_baixa_n", "altura_baixa_n", "altura_adequada_n"] if height else
             ["peso_muito_baixo_n", "peso_baixo_n", "peso_adequado_n", "peso_elevado_n"])
    total = "avaliados_altura" if height else "avaliados_peso"
    count_columns = [p + " - Quantidade" for p in parts] + ["Total"]
    pct_columns = [p + " - %" for p in parts]
    interpreted = [interpret_sisvan_counts([row[c] for c in count_columns],
                    [row[c] for c in pct_columns], row["Código IBGE"]) for _, row in raw.iterrows()]
    table = pd.DataFrame({f"codigo_{source}_original": codes(raw["Código IBGE"], 6, source),
                          f"municipio_{source}": raw["Município"], f"uf_{source}": raw["UF"],
                          f"ano_{source}": metadata["ano"]})
    for i, name in enumerate(names + [total]):
        table[name] = [v[0][i] for v in interpreted]
        table[name + "_valor_fonte"] = raw[count_columns[i]]
    for column, name in zip(pct_columns, names):
        table[name + "_percentual_fonte"] = raw[column]
    table[f"{source}_escala_alterada"] = [v[1] for v in interpreted]
    table[f"{source}_escalas_compativeis"] = [v[2] for v in interpreted]
    if not table[names].sum(axis=1).eq(table[total]).all():
        raise ValueError(f"{source}: soma das categorias difere do total")
    indicator = "dai" if height else "dpi"
    table[indicator + "_n"] = table[names[:2]].sum(axis=1, min_count=2)
    table[indicator + "_pct"] = table[indicator + "_n"].div(table[total].where(table[total] > 0)) * 100
    check_range(table[indicator + "_pct"], 0, 100, indicator)
    return table


def prepare_base(directory):
    directory = Path(directory)
    hashes = verify_hashes(directory)
    ivs = pd.read_csv(directory / DATA_FILES["ivs_idhm"], dtype="string", keep_default_na=False)
    tables = {"ivs_idhm": pd.DataFrame({"codigo_ivs_original": codes(ivs["municipio"], 7, "IVS/IDHM"),
                                        "municipio_ivs": ivs["nome_municipio_uf"]})}
    for column in ivs.columns:
        if column not in ("municipio", "nome_municipio_uf"):
            tables["ivs_idhm"][column] = numeric(ivs[column], column)
            check_range(tables["ivs_idhm"][column], 0, 1, column)
    ci = pd.read_csv(directory / DATA_FILES["cadinsan"], dtype="string", keep_default_na=False)
    tables["cadinsan"] = pd.DataFrame({"codigo_cadinsan_original": codes(ci["Cod_IBGE"], 7, "CadInsan"),
                                      "municipio_cadinsan": ci["Município"]})
    # Este campo pertence ao próprio CSV CadInsan: é seu denominador de famílias.
    tables["cadinsan"]["familias_cadinsan"] = numeric(ci["Cadastros_Cadunico"], "famílias CadInsan")
    for scenario in ("com_PBF", "sem_PBF"):
        for original, name in [(f"Cadinsan_absoluto_{scenario}", f"cadinsan_n_{scenario}"),
                               (f"Cadinsan_proporcional_{scenario}", f"cadinsan_pct_{scenario}_arquivo")]:
            tables["cadinsan"][name] = numeric(ci[original], original)
    for source in ("sisvan_altura", "sisvan_peso"):
        tables[source] = read_nutrition(directory, source)
    for source, table in tables.items():
        original = next(c for c in table if c.startswith("codigo_"))
        table["codigo_ibge_6"] = table[original].str[:6]
        if table["codigo_ibge_6"].duplicated().any():
            raise ValueError(f"{source}: prefixos municipais duplicados")
        table[f"tem_{source}"] = True
    base = tables["sisvan_altura"]
    for source in ("sisvan_peso", "ivs_idhm", "cadinsan"):
        base = base.merge(tables[source], on="codigo_ibge_6", how="outer", validate="one_to_one")
    for source in tables:
        base[f"tem_{source}"] = base[f"tem_{source}"].fillna(False).astype(bool)
    original_7 = ["codigo_cadinsan_original", "codigo_ivs_original"]
    base["codigo_ibge_7"] = base[original_7].bfill(axis=1).iloc[:, 0]
    for column in original_7:
        if (base[column].notna() & base["codigo_ibge_7"].ne(base[column])).any():
            raise ValueError("Conflito entre códigos das fontes sociais")
    # Na ausência de código de sete dígitos, o prefixo original é explicitamente mantido.
    base["codigo_ibge"] = base["codigo_ibge_7"].fillna(base["codigo_ibge_6"])
    base["municipio"] = base[["municipio_sisvan_altura", "municipio_sisvan_peso",
                              "municipio_cadinsan", "municipio_ivs"]].bfill(axis=1).iloc[:, 0]
    base["uf"] = base["codigo_ibge_6"].str[:2].map(lambda c: UF_REGIONS[c][0])
    base["regiao"] = base["codigo_ibge_6"].str[:2].map(lambda c: UF_REGIONS[c][1])
    for source in ("sisvan_altura", "sisvan_peso"):
        present = base[f"tem_{source}"]
        if base.loc[present, "uf"].ne(base.loc[present, f"uf_{source}"]).any():
            raise ValueError(f"{source}: UF incompatível com código municipal")
    counts = [c for c in base if c.endswith("_n") or c.startswith("avaliados_") and not c.endswith("_fonte")]
    counts += ["familias_cadinsan", "cadinsan_n_com_PBF", "cadinsan_n_sem_PBF"]
    for column in counts:
        values = base[column].dropna()
        if (values < 0).any() or not np.isclose(values % 1, 0).all():
            raise ValueError(f"{column}: contagem negativa ou não inteira")
    for scenario in ("com_PBF", "sem_PBF"):
        absolute, denominator = base[f"cadinsan_n_{scenario}"], base["familias_cadinsan"]
        if (absolute > denominator).fillna(False).any():
            raise ValueError("CadInsan absoluto excede denominador")
        calculated = absolute.div(denominator.where(denominator > 0)) * 100
        reported = base[f"cadinsan_pct_{scenario}_arquivo"]
        check_range(reported, 0, 100, "CadInsan na fonte")
        complete = calculated.notna() & reported.notna()
        if not np.allclose(calculated[complete], reported[complete], atol=0.11):
            raise ValueError("Percentual CadInsan diverge da razão no arquivo")
        base[f"cadinsan_pct_{scenario}"] = calculated
        base[f"cadinsan_diferenca_pp_{scenario}"] = calculated - reported
        check_range(calculated, 0, 100, "CadInsan recalculado")
    base["cadinsan_referencia"] = "2025-01"
    height = base["tem_sisvan_altura"]
    audit = pd.DataFrame([{"fonte": source, "registros_origem": len(table),
                          "municipios_na_uniao": len(base),
                          "correspondencias_sisvan": int((base[f"tem_{source}"] & height).sum()),
                          "sisvan_sem_fonte": int((height & ~base[f"tem_{source}"]).sum()),
                          "fonte_sem_sisvan": int((~height & base[f"tem_{source}"]).sum())}
                         for source, table in tables.items()])
    return base.sort_values("codigo_ibge_6").reset_index(drop=True), audit, hashes


def nullable_flag(values, valid, cutoff, operator=">="):
    flag = pd.Series(pd.NA, index=values.index, dtype="boolean")
    comparison = values.ge(cutoff) if operator == ">=" else values.lt(cutoff)
    flag.loc[valid] = comparison.loc[valid]
    return flag


def classify(base, quantile=0.75, min_evaluated=20, scenario="com_PBF", dai_cutoff=6.7, dpi_cutoff=1.8):
    if not 0 < quantile < 1 or not isinstance(min_evaluated, int) or min_evaluated < 1:
        raise ValueError("Quantil deve estar entre 0 e 1; mínimo deve ser inteiro positivo")
    if scenario not in ("com_PBF", "sem_PBF"):
        raise ValueError("Cenário inválido")
    if not np.isfinite(dpi_cutoff) or not 0 <= dpi_cutoff <= 100:
        raise ValueError("Corte DPI deve ser percentual entre 0 e 100")
    result = base.copy()
    result["cadinsan_pct"] = result[f"cadinsan_pct_{scenario}"]
    result["cadinsan_pct_arquivo"] = result[f"cadinsan_pct_{scenario}_arquivo"]
    result["cadinsan_n"] = result[f"cadinsan_n_{scenario}"]
    valid_ci = result["cadinsan_pct"].notna() & result["familias_cadinsan"].gt(0)
    valid_height = result["dai_pct"].notna() & result["avaliados_altura"].ge(min_evaluated)
    valid_weight = result["dpi_pct"].notna() & result["avaliados_peso"].ge(min_evaluated)
    if not valid_ci.any():
        raise ValueError("Nenhum município válido para quantil CadInsan")
    ci_cut = result.loc[valid_ci, "cadinsan_pct"].quantile(quantile)
    if dai_cutoff == "P75":
        if not valid_height.any():
            raise ValueError("Nenhum município válido para quantil DAI")
        height_cut = result.loc[valid_height, "dai_pct"].quantile(0.75)
        dai_method = "quantil exploratório"
    else:
        height_cut, dai_method = float(dai_cutoff), "referência Mapa InSAN" if float(dai_cutoff) == 6.7 else "referência OMS"
        if not np.isfinite(height_cut) or not 0 <= height_cut <= 100:
            raise ValueError("Corte DAI deve ser percentual entre 0 e 100")
    specs = [("ivs", "ivs", result["ivs"].notna(), 0.401, ">=", "faixa IVS/Ipea"),
             ("idhm", "idhm", result["idhm"].notna(), 0.600, "<", "baixo desenvolvimento adotado"),
             ("cadinsan", "cadinsan_pct", valid_ci, ci_cut, ">=", "quantil exploratório"),
             ("dai", "dai_pct", valid_height, height_cut, ">=", dai_method),
             ("dpi", "dpi_pct", valid_weight, dpi_cutoff, ">=", "referência Mapa InSAN complementar")]
    cuts = []
    for label, column, valid, cutoff, operator, method in specs:
        result[f"criterio_{label}"] = nullable_flag(result[column], valid, cutoff, operator)
        cuts.append({"indicador": label, "corte": float(cutoff), "operador": operator,
                     "metodo": method, "municipios_referencia": int(valid.sum()),
                     "municipios_no_criterio": int(result[f"criterio_{label}"].sum()),
                     "percentil": quantile * 100 if label == "cadinsan" else 75 if label == "dai" and dai_cutoff == "P75" else None})
    # Mesmo com um critério conhecido desfavorável, falta de outro impede a dimensão social.
    valid_social = result[["criterio_ivs", "criterio_idhm"]].notna().all(axis=1)
    result["dim_social"] = (result["criterio_ivs"] & result["criterio_idhm"]).where(valid_social)
    result["dim_alimentar"] = result["criterio_cadinsan"]
    result["dim_nutricional"] = result["criterio_dai"]
    result["social_alta"] = result["dim_social"]
    result["cadinsan_alto"] = result["dim_alimentar"]
    result["nutricional_alta"] = result["dim_nutricional"]
    result["social_status"] = "dados insuficientes"
    for ivs_flag, idhm_flag, label in [(True, True, "ambos desfavoráveis"),
                                     (True, False, "apenas IVS desfavorável"),
                                     (False, True, "apenas IDHM desfavorável"),
                                     (False, False, "nenhum desfavorável")]:
        mask = valid_social & result["criterio_ivs"].eq(ivs_flag) & result["criterio_idhm"].eq(idhm_flag)
        result.loc[mask.fillna(False), "social_status"] = label
    result["elegivel_principal"] = result[DIMENSIONS].notna().all(axis=1)
    result["n_dimensoes_desfavoraveis"] = result[DIMENSIONS].sum(axis=1).where(result["elegivel_principal"]).astype("Int64")
    result["convergencia_total"] = result["n_dimensoes_desfavoraveis"].eq(3).astype("boolean")
    result["prioridade_maxima"] = result["convergencia_total"]
    result["classe_prioridade"] = result["n_dimensoes_desfavoraveis"].map(PRIORITY_LABELS).fillna("dados insuficientes")
    result["dai_categoria_oms"] = pd.cut(result["dai_pct"].where(valid_height),
        [-np.inf, 2.5, 10, 20, 30, np.inf], labels=OMS_LABELS, right=False).astype("string").fillna("dados insuficientes")
    both_valid = valid_height & valid_weight
    result["nutricional_mapa_insan"] = (nullable_flag(result["dai_pct"], valid_height, 6.7)
                                          & result["criterio_dpi"]).where(both_valid)
    result["convergencia_total_dai_dpi"] = (result["dim_social"] & result["dim_alimentar"]
                                             & result["nutricional_mapa_insan"]).where(valid_social & valid_ci & both_valid)
    result["dpi_status"] = result["criterio_dpi"].map({True: "DPI ≥ referência", False: "DPI abaixo da referência"}).fillna("dados insuficientes")
    reasons = []
    for row in result.itertuples():
        missing = []
        if pd.isna(row.ivs): missing.append("IVS ausente")
        if pd.isna(row.idhm): missing.append("IDHM ausente")
        if pd.isna(row.cadinsan_pct) or not row.familias_cadinsan > 0:
            missing.append("CadInsan ausente ou denominador inválido")
        if pd.isna(row.dai_pct): missing.append("DAI ausente ou sem avaliações")
        elif row.avaliados_altura < min_evaluated: missing.append(f"menos de {min_evaluated} avaliações de altura")
        reasons.append("; ".join(missing))
    result["motivo_nao_classificacao"] = reasons
    result["cenario_cadinsan"] = scenario
    result["quantil_cadinsan"] = quantile
    result["minimo_avaliados"] = min_evaluated
    return result, pd.DataFrame(cuts)


def compare_rounding(base, reference, quantile=0.75, min_evaluated=20, scenario="com_PBF", dai_cutoff=6.7):
    rounded = base.copy()
    for option in ("com_PBF", "sem_PBF"):
        rounded[f"cadinsan_pct_{option}"] = rounded[f"cadinsan_pct_{option}_arquivo"]
    previous, cuts = classify(rounded, quantile, min_evaluated, scenario, dai_cutoff)
    comparison = reference[["codigo_ibge_6", "municipio", "uf", "cadinsan_pct", "cadinsan_pct_arquivo"]].copy()
    comparison["selecionado_percentual_csv"] = previous["convergencia_total"]
    comparison["selecionado_sem_arredondamento"] = reference["convergencia_total"]
    comparison["mudou_selecao"] = comparison["selecionado_percentual_csv"].fillna(False).ne(comparison["selecionado_sem_arredondamento"].fillna(False))
    return comparison, cuts


def sensitivity(base, reference, quantiles=(0.70, 0.75, 0.80, 0.90), minima=(20, 50, 100),
                scenarios=("com_PBF", "sem_PBF"), dai_cuts=(6.7, 10.0, "P75")):
    ref = set(reference.loc[reference["convergencia_total"].fillna(False), "codigo_ibge_6"])
    records, selections = [], []
    for scenario, quantile, minimum, dai_cut in product(scenarios, quantiles, minima, dai_cuts):
        result, cuts = classify(base, quantile, minimum, scenario, dai_cut)
        selected = set(result.loc[result["convergencia_total"].fillna(False), "codigo_ibge_6"])
        union = ref | selected
        records.append({"cenario": scenario, "quantil": quantile, "minimo_avaliados": minimum,
                        "referencia_dai": str(dai_cut), "corte_dai_pct": float(cuts.set_index("indicador").loc["dai", "corte"]),
                        "corte_cadinsan_pct": float(cuts.set_index("indicador").loc["cadinsan", "corte"]),
                        "elegiveis": int(result["elegivel_principal"].sum()),
                        "municipios_dim_alimentar": int(result["dim_alimentar"].sum()),
                        "municipios_dim_nutricional": int(result["dim_nutricional"].sum()),
                        "convergencia_total": len(selected), "entram_vs_principal": len(selected - ref),
                        "saem_vs_principal": len(ref - selected), "coincidentes_principal": len(ref & selected),
                        "jaccard_principal": len(ref & selected) / len(union) if union else 1.0})
        selections.append(pd.DataFrame({"codigo_ibge_6": result["codigo_ibge_6"],
                                         "elegivel": result["elegivel_principal"],
                                         "selecionado": result["convergencia_total"].fillna(False).astype(int)}))
    stability = pd.concat(selections).groupby("codigo_ibge_6").agg(
        cenarios_elegiveis=("elegivel", "sum"), cenarios_selecionado=("selecionado", "sum"))
    stability["cenarios_testados"] = len(records)
    stability["fracao_cenarios_selecionado"] = stability["cenarios_selecionado"] / len(records)
    return pd.DataFrame(records), stability.reset_index()


def geographic_summary(result, geography="regiao"):
    records = []
    for area, table in result.groupby(geography, sort=True):
        selected = int(table["convergencia_total"].sum())
        eligible = int(table["elegivel_principal"].sum())
        record = {geography: area, "municipios": len(table), "elegiveis": eligible,
                  "convergencia_total": selected, "pct_convergencia_universo": selected / len(table) * 100,
                  "pct_convergencia_elegiveis": selected / eligible * 100 if eligible else np.nan}
        for label, total in [("dai", "avaliados_altura"), ("dpi", "avaliados_peso")]:
            valid = table[total].gt(0) & table[label + "_n"].notna()
            denominator = table.loc[valid, total].sum()
            record[total] = denominator
            record[label + "_n"] = table.loc[valid, label + "_n"].sum()
            record[label + "_pct_agregado"] = record[label + "_n"] / denominator * 100 if denominator else np.nan
        records.append(record)
    return pd.DataFrame(records)


def nutrition_comparison(result):
    common = result["nutricional_mapa_insan"].notna()
    records = []
    for label, valid in [("Todos os válidos para DAI", result["dim_nutricional"].notna()),
                         ("Denominadores válidos em altura e peso", common)]:
        records.append({"universo": label, "municipios": int(valid.sum()),
                        "dai_isolado": int(result.loc[valid, "dim_nutricional"].sum()),
                        "dai_e_dpi": int(result.loc[valid, "nutricional_mapa_insan"].sum()),
                        "convergencia_dai": int(result.loc[valid, "convergencia_total"].sum()),
                        "convergencia_dai_dpi": int(result.loc[valid, "convergencia_total_dai_dpi"].sum())})
    return pd.DataFrame(records)


def distribution_figure(result, output):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    columns = [("ivs", "IVS — 2010"), ("idhm", "IDHM — 2010"), ("cadinsan_pct", "CadInsan (%)"),
               ("dai_pct", "DAI (%) — com avaliações"), ("dpi_pct", "DPI (%) — com avaliações"),
               ("avaliados_altura", "Avaliações de altura — escala log")]
    for ax, (column, title) in zip(axes.flat, columns):
        values = result[column].dropna()
        if column == "avaliados_altura":
            values = np.log10(values[values > 0])
            ax.set_xlabel("log10(total avaliado)")
        ax.hist(values, bins=35, color="#327c81", edgecolor="white", linewidth=0.4)
        ax.set(title=title, ylabel="Municípios")
    fig.tight_layout()
    save_figure(fig, output, "01_distribuicoes")
    return fig


def association_figure(result, thresholds, output):
    import matplotlib.pyplot as plt
    cuts = thresholds.set_index("indicador")["corte"]
    valid = result[result["elegivel_principal"]]
    chosen = valid["convergencia_total"].fillna(False)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (column, label, criterion) in zip(axes, [("ivs", "IVS (2010)", "ivs"),
        ("idhm", "IDHM (2010)", "idhm"), ("cadinsan_pct", "CadInsan (%)", "cadinsan")]):
        ax.scatter(valid[column], valid["dai_pct"], s=8, alpha=0.25, color="#327c81")
        ax.scatter(valid.loc[chosen, column], valid.loc[chosen, "dai_pct"], s=13, alpha=0.7, color="#9e1b32")
        ax.axhline(cuts["dai"], color="#777777", linestyle="--", linewidth=1)
        ax.axvline(cuts[criterion], color="#777777", linestyle="--", linewidth=1)
        ax.set(xlabel=label, ylabel="DAI (%)", title=f"{label} × DAI")
    fig.suptitle("Municípios elegíveis; vermelho = convergência das três dimensões")
    fig.tight_layout()
    save_figure(fig, output, "02_associacoes")
    return fig


def correlation_figure(result, output, minimum=20):
    import matplotlib.pyplot as plt
    cols = ["ivs", "idhm", "cadinsan_pct", "dai_pct", "dpi_pct"]
    data = result[cols].copy()
    for indicator, total in [("dai_pct", "avaliados_altura"), ("dpi_pct", "avaliados_peso")]:
        data[indicator] = data[indicator].where(result[total].ge(minimum))
    corr = data.corr(method="spearman", min_periods=3)
    pairs = data.notna().astype(int).T @ data.notna().astype(int)
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(5), ["IVS", "IDHM", "CadInsan", "DAI", "DPI"])
    ax.set_yticks(range(5), ["IVS", "IDHM", "CadInsan", "DAI", "DPI"])
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}\nn={pairs.iloc[i, j]}", ha="center", va="center",
                    fontsize=9, color="white" if abs(corr.iloc[i, j]) > 0.65 else "black")
    ax.set_title("Spearman — pares disponíveis; nutrição com denominador mínimo")
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    save_figure(fig, output, "03_correlacoes")
    return fig, corr, pairs


def sensitivity_figure(table, output, scenario="com_PBF"):
    import matplotlib.pyplot as plt
    subset = table.loc[table["cenario"].eq(scenario)]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    for ax, reference in zip(axes, ["6.7", "10.0", "P75"]):
        for minimum, group in subset.loc[subset["referencia_dai"].eq(reference)].groupby("minimo_avaliados"):
            group = group.sort_values("quantil")
            ax.plot(group["quantil"] * 100, group["convergencia_total"], marker="o", label=f"n ≥ {minimum}")
        ax.set(title=f"DAI ≥ {reference.replace('.', ',')}{'%' if reference != 'P75' else ''}",
               xlabel="Percentil CadInsan", xticks=[70, 75, 80, 90])
        ax.legend()
    axes[0].set_ylabel("Municípios com convergência total")
    fig.suptitle(f"Sensibilidade — {scenario}; cortes IVS/IDHM fixos")
    fig.tight_layout()
    save_figure(fig, output, "04_sensibilidade")
    return fig


def dictionary():
    rows = [
        ("codigo_ibge", "Código de sete dígitos disponível nas fontes sociais; prefixo de seis se desconhecido", "texto"),
        ("codigo_ibge_6", "Prefixo único usado na integração; não valida limites históricos", "texto"),
        ("codigo_ibge_7", "Código das fontes sociais; sem inventar dígito verificador", "texto"),
        ("municipio / uf / regiao", "Identificação territorial", "texto"),
        ("ivs", "Índice de Vulnerabilidade Social, 2010; corte ≥ 0,401", "0–1"),
        ("idhm", "Desenvolvimento humano municipal, 2010; corte < 0,600", "0–1"),
        ("familias_cadinsan", "Denominador familiar do próprio CSV CadInsan, janeiro/2025", "famílias"),
        ("cadinsan_n", "Famílias em risco estimado no cenário selecionado", "famílias"),
        ("cadinsan_pct", "100 × cadinsan_n / familias_cadinsan; corte P75 exploratório", "%"),
        ("cadinsan_pct_*_arquivo", "Percentuais oficiais arredondados, preservados", "%"),
        ("cadinsan_diferenca_pp_*", "Diferença da razão recalculada para o percentual da fonte", "pontos percentuais"),
        ("avaliados_altura / avaliados_peso", "Totais separados por relatório SISVAN, menores de cinco anos, 2025", "avaliados"),
        ("altura_muito_baixa_n / altura_baixa_n", "Categorias de baixa estatura classificadas pelo SISVAN", "avaliados"),
        ("peso_muito_baixo_n / peso_baixo_n", "Categorias de baixo peso classificadas pelo SISVAN", "avaliados"),
        ("dai_n / dpi_n", "Soma das duas categorias desfavoráveis do respectivo relatório", "avaliados"),
        ("dai_pct / dpi_pct", "100 × numerador / respectivo total; total zero gera ausência", "%"),
        ("dai_categoria_oms", "Faixas <2,5; [2,5,10); [10,20); [20,30); ≥30; insuficiente abaixo do mínimo", "categoria"),
        ("criterio_*", "Critérios individuais anuláveis; DPI não integra seleção principal", "booleano anulável"),
        ("dim_social / social_alta", "IVS ≥ 0,401 E IDHM < 0,600; ausente se falta algum", "booleano anulável"),
        ("social_status", "Combinação dos dois critérios sociais; não representa duas dimensões", "categoria"),
        ("dim_alimentar / cadinsan_alto", "CadInsan ≥ P75 nacional no cenário selecionado", "booleano anulável"),
        ("dim_nutricional / nutricional_alta", "DAI ≥ 6,7%, com total altura ≥20 no cenário principal", "booleano anulável"),
        ("nutricional_mapa_insan", "DAI ≥6,7% E DPI ≥1,8%, ambos com denominadores ≥mínimo; adaptação, não Mapa oficial", "booleano anulável"),
        ("dpi_status", "Situação complementar do DPI, com mínimo próprio de peso", "categoria"),
        ("n_dimensoes_desfavoraveis", "Número de dimensões desfavoráveis, ausente se qualquer dimensão é desconhecida", "0–3"),
        ("classe_prioridade", "Classe descritiva da convergência; não diagnóstico ou ranking", "categoria"),
        ("convergencia_total / prioridade_maxima", "As três dimensões desfavoráveis; ausente se informação insuficiente", "booleano anulável"),
        ("convergencia_total_dai_dpi", "Social E alimentar E critério nutricional combinado DAI+DPI", "booleano anulável"),
        ("motivo_nao_classificacao", "Motivo de insuficiência para a seleção principal; DPI não é requisito", "texto"),
        ("*_valor_fonte / *_percentual_fonte", "Células da fonte antes de qualquer leitura analítica", "texto"),
        ("sisvan_*_escala_alterada / sisvan_*_escalas_compativeis", "Auditoria de leitura das contagens já existente na v1", "booleano / inteiro"),
        ("tem_*", "Presença de registro na respectiva entrada", "booleano"),
        ("fracao_cenarios_selecionado", "Fração de cenários testados que seleciona; não probabilidade de risco", "0–1"),
    ]
    return pd.DataFrame(rows, columns=["variavel", "definicao", "unidade"])

## 3. Integração, códigos IBGE e qualidade dos dados

A união externa preserva todos os municípios encontrados, inclusive aqueles
sem alguma fonte. O SISVAN usa códigos de seis dígitos; as fontes sociais,
sete. Usa-se o prefixo único de seis, verificando duplicatas e conflitos de
códigos e UFs. `codigo_ibge_7` conserva o código social quando disponível;
`codigo_ibge` mantém o prefixo original se o sétimo dígito não é conhecido.
Nenhum dígito verificador é inventado.

As verificações de integridade já existentes (hashes, esquema, categorias,
soma das contagens e leitura dos percentuais) são preservadas. Não se acrescenta
uma seção de comparação entre DAI por contagens e DAI por percentuais oficiais.
Ausência de informação nunca é substituída por zero.

In [5]:
validacao_catalogo, proveniencia_catalogo = validate_catalogue(
    PASTA_DADOS / "documentacao/dataset_catalogue.json", PASTA_DADOS)
base, controle_integracao, hashes_entrada = prepare_base(PASTA_DADOS)
figure_style()
display(tabela_br(controle_integracao))
colunas_fontes = ["tem_ivs_idhm", "tem_cadinsan", "tem_sisvan_altura", "tem_sisvan_peso"]
ausencias = base.loc[~base[colunas_fontes].all(axis=1),
    ["codigo_ibge", "municipio", "uf"] + colunas_fontes]
display(tabela_br(ausencias))
qualidade_sisvan = pd.DataFrame([
    {"relatorio": label, "controle": control, "municipios": int(mask.sum())}
    for label, total, indicator in [("Altura X Idade", "avaliados_altura", "dai_pct"),
                                     ("Peso X Idade", "avaliados_peso", "dpi_pct")]
    for control, mask in [("Sem avaliações", base[total].eq(0)),
                          ("Abaixo do mínimo principal", base[total].between(1, MINIMO_AVALIADOS - 1)),
                          ("Indicador ausente", base[indicator].isna())]])
display(tabela_br(qualidade_sisvan))
interpretacao_sisvan = base[["codigo_ibge", "municipio", "uf"] +
    [c for c in base if c.endswith(("_valor_fonte", "_percentual_fonte", "_escala_alterada", "_escalas_compativeis"))]].copy()

,fonte,registros_origem,municipios_na_uniao,correspondencias_sisvan,sisvan_sem_fonte,fonte_sem_sisvan
0,ivs_idhm,5.565,5.571,5.565,6,0
1,cadinsan,5.570,5.571,5.570,1,0
2,sisvan_altura,5.571,5.571,5.571,0,0
3,sisvan_peso,5.571,5.571,5.571,0,0


,codigo_ibge,municipio,uf,tem_ivs_idhm,tem_cadinsan,tem_sisvan_altura,tem_sisvan_peso
224,1504752,MOJUI DOS CAMPOS,PA,False,True,True,True
4503,4212650,PESCARIA BRAVA,SC,False,True,True,True
4605,4220000,BALNEARIO RINCAO,SC,False,True,True,True
4923,4314548,PINTO BANDEIRA,RS,False,True,True,True
5160,5006275,PARAISO DAS AGUAS,MS,False,True,True,True
5199,510183,BOA ESPERANCA DO NORTE,MT,False,False,True,True


,relatorio,controle,municipios
0,Altura X Idade,Sem avaliações,1
1,Altura X Idade,Abaixo do mínimo principal,4
2,Altura X Idade,Indicador ausente,1
3,Peso X Idade,Sem avaliações,1
4,Peso X Idade,Abaixo do mínimo principal,4
5,Peso X Idade,Indicador ausente,1


## 4. Construção dos indicadores

$$CadInsan(\%)=100\times\frac{\text{famílias em risco estimado no cenário escolhido}}
{\text{famílias no universo analisado pelo CadInsan}}$$

$$DAI(\%)=100\times\frac{N(\text{altura muito baixa para idade})+N(\text{altura baixa para idade})}
{N(\text{avaliados em altura por idade})}$$

$$DPI(\%)=100\times\frac{N(\text{peso muito baixo para idade})+N(\text{peso baixo para idade})}
{N(\text{avaliados em peso por idade})}$$

DAI é o **déficit de altura para idade** e DPI, o **déficit de peso para idade**,
em crianças de 0 a menos de cinco anos. Utilizam-se as categorias já
classificadas pelo SISVAN, sem calcular escores-Z ou ligar microdados individuais.
Os denominadores de altura e peso são mantidos separados.

CadInsan é recalculado sem arredondamento; seus percentuais de origem ficam
em colunas próprias. Denominador zero produz indicador indefinido (`NaN`),
não percentual zero. Os índices IVS e IDHM permanecem como publicados.
A tabela abaixo descreve a união das fontes, antes da seleção.

In [6]:
descritivas = base[["ivs", "idhm", "cadinsan_pct_com_PBF", "cadinsan_pct_sem_PBF",
                   "dai_pct", "dpi_pct", "avaliados_altura", "avaliados_peso"]].describe().T
display(tabela_br(descritivas))
totais_nutricionais = pd.DataFrame([
    {"indicador": label.upper(), "numerador": base[label + "_n"].sum(),
     "denominador": base.loc[base[label + "_n"].notna(), total].sum(),
     "percentual_agregado": base[label + "_n"].sum() /
        base.loc[base[label + "_n"].notna(), total].sum() * 100}
    for label, total in [("dai", "avaliados_altura"), ("dpi", "avaliados_peso")]])
display(tabela_br(totais_nutricionais))

,count,mean,std,min,25%,50%,75%,max
ivs,"5.565,000","0,352","0,130","0,090","0,246","0,335","0,448","0,784"
idhm,"5.565,000","0,659","0,072","0,418","0,599","0,665","0,718","0,862"
cadinsan_pct_com_PBF,"5.570,000","8,688","4,716","0,000","5,362","7,766","10,970","33,084"
cadinsan_pct_sem_PBF,"5.570,000","18,738","11,556","0,000","11,224","16,303","22,877","83,869"
dai_pct,"5.570,000","10,202","4,905","0,000","6,929","9,400","12,375","52,744"
dpi_pct,"5.570,000","2,944","1,908","0,000","1,896","2,630","3,533","30,587"
avaliados_altura,"5.571,000","1.408,439","5.613,397","0,000","255,000","548,000","1.223,500","257.018,000"
avaliados_peso,"5.571,000","1.408,434","5.612,079","0,000","255,000","548,000","1.223,500","256.853,000"


,indicador,numerador,denominador,percentual_agregado
0,DAI,877.708,7.846.413,"11,19%"
1,DPI,266.641,7.846.386,"3,40%"


**Interpretação:** o percentual agregado é a razão entre somas, não a média
municipal. Muitos acompanhamentos não garantem representatividade. Sem
denominador demográfico compatível de menores de cinco anos, não se calcula
cobertura populacional. Altura e peso não devem ser somados como pessoas distintas.

## 5. As três dimensões e a sobreposição

| Dimensão | Condição desfavorável principal |
|---|---|
| Social (`dim_social`) | IVS ≥ 0,401 **e** IDHM < 0,600 |
| Alimentar (`dim_alimentar`) | CadInsan ≥ P75 nacional |
| Nutricional (`dim_nutricional`) | DAI ≥ 6,7% com ≥20 avaliações de altura |

O mínimo de 20 reduz o uso de denominadores extremamente pequenos, mas não
elimina instabilidade: com 20 avaliados, uma criança altera o percentual em
cinco pontos. Por isso são testados mínimos de 50 e 100.

A quantidade de dimensões desfavoráveis varia de **0 a 3**, somente quando
todas são conhecidas. Os rótulos são descritivos e não uma classificação oficial:
0 = sem sinais pelos critérios adotados; 1 = atenção; 2 = prioridade alta;
3 = prioridade muito alta. **Zero não significa ausência de insegurança alimentar**;
significa que nenhum dos três critérios dimensionais foi satisfeito.
Uma dimensão desconhecida implica **dados insuficientes** na classificação geral,
sem ocultar os sinais conhecidos nas demais flags.

DPI ausente ou com poucas avaliações impede apenas sua classificação
complementar e a comparação DAI+DPI, não a seleção principal baseada em DAI.

In [7]:
classificados, cortes = classify(base, QUANTIL_CADINSAN, MINIMO_AVALIADOS,
                                 CENARIO_CADINSAN, CORTE_DAI, CORTE_DPI)
display(tabela_br(cortes))
resumo_social = classificados["social_status"].value_counts().rename_axis("social_status").reset_index(name="municipios")
display(tabela_br(resumo_social))
classes = classificados["classe_prioridade"].value_counts().reindex(
    list(PRIORITY_LABELS.values()) + ["dados insuficientes"], fill_value=0)
resumo_classes = classes.rename_axis("classe_prioridade").reset_index(name="municipios")
display(tabela_br(resumo_classes))
sem_classificacao = classificados.loc[~classificados["elegivel_principal"],
    ["codigo_ibge", "municipio", "uf", "avaliados_altura", "motivo_nao_classificacao"]]
display(tabela_br(sem_classificacao))
comparacao_arredondamento, cortes_percentuais_csv = compare_rounding(
    base, classificados, QUANTIL_CADINSAN, MINIMO_AVALIADOS, CENARIO_CADINSAN, CORTE_DAI)
total_com_csv = int(comparacao_arredondamento["selecionado_percentual_csv"].sum())
total_recalculado = int(classificados["convergencia_total"].sum())
display(Markdown(f"**Arredondamento CadInsan:** {total_com_csv} selecionados usando os percentuais "
                 f"do CSV; {total_recalculado} usando as razões sem arredondamento."))
display(tabela_br(comparacao_arredondamento.loc[comparacao_arredondamento["mudou_selecao"]]))

,indicador,corte,operador,metodo,municipios_referencia,municipios_no_criterio,percentil
0,ivs,"0,401",>=,faixa IVS/Ipea,5.565,1.981,—
1,idhm,"0,600",<,baixo desenvolvimento adotado,5.565,1.399,—
2,cadinsan,"10,97%",>=,quantil exploratório,5.570,1.393,75
3,dai,"6,70%",>=,referência Mapa InSAN,5.566,4.292,—
4,dpi,"1,80%",>=,referência Mapa InSAN complementar,5.566,4.329,—


,social_status,municipios
0,nenhum desfavorável,3.487
1,ambos desfavoráveis,1.302
2,apenas IVS desfavorável,679
3,apenas IDHM desfavorável,97
4,dados insuficientes,6


,classe_prioridade,municipios
0,Sem sinais pelos critérios adotados,1.045
1,Atenção — uma dimensão,2.598
2,Prioridade alta — duas dimensões,1.370
3,Prioridade muito alta — três dimensões,548
4,dados insuficientes,10


,codigo_ibge,municipio,uf,avaliados_altura,motivo_nao_classificacao
224,1504752,MOJUI DOS CAMPOS,PA,1.771,IVS ausente; IDHM ausente
4320,4200754,ALTO BELA VISTA,SC,19,menos de 20 avaliações de altura
4503,4212650,PESCARIA BRAVA,SC,416,IVS ausente; IDHM ausente
4605,4220000,BALNEARIO RINCAO,SC,886,IVS ausente; IDHM ausente
4814,4310462,IPIRANGA DO SUL,RS,10,menos de 20 avaliações de altura
4923,4314548,PINTO BANDEIRA,RS,113,IVS ausente; IDHM ausente
4976,4317251,SANTA TEREZA,RS,6,menos de 20 avaliações de altura
5077,4322350,UNIAO DA SERRA,RS,15,menos de 20 avaliações de altura
5160,5006275,PARAISO DAS AGUAS,MS,375,IVS ausente; IDHM ausente
5199,510183,BOA ESPERANCA DO NORTE,MT,0,IVS ausente; IDHM ausente; CadInsan ausente ou denominador inválido; DAI ausente ou sem avaliações


**Arredondamento CadInsan:** 550 selecionados usando os percentuais do CSV; 548 usando as razões sem arredondamento.

,codigo_ibge_6,municipio,uf,cadinsan_pct,cadinsan_pct_arquivo,selecionado_percentual_csv,selecionado_sem_arredondamento,mudou_selecao
1734,270790,SANTA LUZIA DO NORTE,AL,"10,96%","11,00%",True,False,True
2122,292410,PEDRAO,BA,"10,96%","11,00%",True,False,True


## 6. Convergência total e distribuição geográfica

`convergencia_total = dim_social & dim_alimentar & dim_nutricional`.
A lista responde onde os sinais das três dimensões coincidem. Ordena-se
por UF e município, sem ranking. O percentual nacional usa como denominador
**os municípios presentes na malha de apoio do IBGE**; também se informa
o percentual entre elegíveis e eventuais selecionados fora dessa malha.
O universo é explicitado, sem presumir que todos os períodos têm o mesmo
número de municípios. As tabelas por UF/região têm denominadores próprios.

In [8]:
caminho_malha = PASTA_DADOS / "apoio/ibge/malha_municipal_simplificada.geojson"
metadados_malha = json.loads((PASTA_DADOS / "apoio/ibge/malha_municipal_simplificada.metadados.json").read_text())
if hashlib.sha256(caminho_malha.read_bytes()).hexdigest() != metadados_malha["sha256"]:
    raise ValueError("Hash da malha diferente do registrado")
geometria = json.loads(caminho_malha.read_text())
controle_geometria = geometry_audit(geometria, base)
universo_ibge = {str(f["properties"]["codarea"])[:6] for f in geometria["features"]}
convergentes = classificados.loc[classificados["convergencia_total"].fillna(False)].sort_values(["uf", "municipio"])
selecionados_na_malha = int(convergentes["codigo_ibge_6"].isin(universo_ibge).sum())
elegiveis = int(classificados["elegivel_principal"].sum())
resumo_nacional = pd.DataFrame([{
    "municipios": len(base), "municipios_malha_ibge": len(universo_ibge), "elegiveis": elegiveis,
    "convergencia_total": len(convergentes), "selecionados_na_malha": selecionados_na_malha,
    "selecionados_fora_malha": len(convergentes) - selecionados_na_malha,
    "pct_convergencia_brasil_malha": selecionados_na_malha / len(universo_ibge) * 100,
    "pct_convergencia_elegiveis": len(convergentes) / elegiveis * 100 if elegiveis else np.nan}])
display(tabela_br(resumo_nacional))
colunas_selecao = {"municipio": "Município", "uf": "UF", "ivs": "IVS", "idhm": "IDHM",
    "cadinsan_pct": "CadInsan (%)", "dai_pct": "DAI (%)", "avaliados_altura": "Avaliações de altura",
    "familias_cadinsan": "Famílias no universo CadInsan", "cadinsan_n": "Famílias em risco estimado"}
display(tabela_br(convergentes[list(colunas_selecao)].rename(columns=colunas_selecao).head(30)))
print("Prévia de 30 municípios; a lista completa será exportada.")
resumo_uf = geographic_summary(classificados, "uf")
resumo_regional = geographic_summary(classificados, "regiao")
display(tabela_br(resumo_uf))
display(tabela_br(resumo_regional))

,municipios,municipios_malha_ibge,elegiveis,convergencia_total,selecionados_na_malha,selecionados_fora_malha,pct_convergencia_brasil_malha,pct_convergencia_elegiveis
0,5.571,5.571,5.561,548,548,0,"9,84%","9,85%"


,Município,UF,IVS,IDHM,CadInsan (%),DAI (%),Avaliações de altura,Famílias no universo CadInsan,Famílias em risco estimado
53,ASSIS BRASIL,AC,"0,500","0,588","14,88%","26,86%",1.158,1.499,223
56,CAPIXABA,AC,"0,469","0,575","12,67%","10,04%",667,1.894,240
62,MANOEL URBANO,AC,"0,589","0,551","14,87%","20,34%",1.239,3.006,447
63,MARECHAL THAUMATURGO,AC,"0,610","0,501","11,00%","19,79%",1.804,2.410,265
73,PORTO ACRE,AC,"0,492","0,576","18,92%","9,36%",1.228,4.492,850
67,RODRIGUES ALVES,AC,"0,553","0,567","13,61%","16,37%",959,3.013,410
68,SANTA ROSA DO PURUS,AC,"0,564","0,517","11,46%","49,71%",690,1.012,116
1654,BARRA DE SANTO ANTONIO,AL,"0,551","0,557","11,73%","13,32%",1.036,2.891,339
1668,CARNEIROS,AL,"0,604","0,526","12,44%","18,60%",758,2.210,275
1669,CHA PRETA,AL,"0,465","0,575","13,44%","10,45%",488,1.771,238


Prévia de 30 municípios; a lista completa será exportada.


,uf,municipios,elegiveis,convergencia_total,pct_convergencia_universo,pct_convergencia_elegiveis,avaliados_altura,dai_n,dai_pct_agregado,avaliados_peso,dpi_n,dpi_pct_agregado
0,AC,22,22,7,"31,82%","31,82%",46.322,7.069,"15,26%",46.319,1.816,"3,92%"
1,AL,102,102,19,"18,63%","18,63%",174.460,19.725,"11,31%",174.461,5.024,"2,88%"
2,AM,62,62,32,"51,61%","51,61%",272.261,44.844,"16,47%",272.259,12.015,"4,41%"
3,AP,16,16,3,"18,75%","18,75%",42.447,6.688,"15,76%",42.447,1.725,"4,06%"
4,BA,417,417,105,"25,18%","25,18%",618.042,65.399,"10,58%",618.053,21.529,"3,48%"
5,CE,184,184,24,"13,04%","13,04%",425.477,45.581,"10,71%",425.479,10.454,"2,46%"
6,DF,1,1,0,"0,00%","0,00%",89.204,8.458,"9,48%",89.203,3.180,"3,56%"
7,ES,78,78,0,"0,00%","0,00%",142.102,13.213,"9,30%",142.107,3.881,"2,73%"
8,GO,246,246,0,"0,00%","0,00%",247.522,29.440,"11,89%",247.522,9.281,"3,75%"
9,MA,217,217,48,"22,12%","22,12%",367.637,54.487,"14,82%",367.647,19.814,"5,39%"


,regiao,municipios,elegiveis,convergencia_total,pct_convergencia_universo,pct_convergencia_elegiveis,avaliados_altura,dai_n,dai_pct_agregado,avaliados_peso,dpi_n,dpi_pct_agregado
0,Centro-Oeste,468,466,3,"0,64%","0,64%",631.574,67.995,"10,77%",631.571,21.082,"3,34%"
1,Nordeste,1.794,1.794,406,"22,63%","22,63%",2.568.298,297.697,"11,59%",2.568.343,89.064,"3,47%"
2,Norte,450,449,127,"28,22%","28,29%",1.022.788,148.137,"14,48%",1.022.793,42.435,"4,15%"
3,Sudeste,1.668,1.668,11,"0,66%","0,66%",2.600.965,268.290,"10,32%",2.600.886,87.098,"3,35%"
4,Sul,1.191,1.184,1,"0,08%","0,08%",1.022.788,95.589,"9,35%",1.022.793,26.962,"2,64%"


**Interpretação:** diferenciar a quantidade selecionada do percentual em cada
área. Uma região com mais municípios pode ter mais selecionados sem maior
proporção. DAI/DPI agregados das tabelas regionais usam todos os registros
válidos da área, não apenas os selecionados nem apenas aqueles acima do mínimo.
Não são prevalências representativas de todas as crianças residentes.

## 7. DAI por faixas OMS e comparação com DPI

As [faixas de prevalência de baixa estatura da OMS/UNICEF](https://www.who.int/data/nutrition/nlis/info/malnutrition-in-children)
permitem descrever a magnitude do DAI: **<2,5% muito baixa; 2,5 a <10% baixa;
10 a <20% média; 20 a <30% alta; ≥30% muito alta**. São categorias de
prevalência, não diagnósticos individuais. Sua aplicação aos acompanhados pelo
SISVAN é descritiva, sem assegurar representatividade municipal.

Essas faixas não substituem o critério principal de **6,7%**; este pode ser
atendido dentro da categoria OMS “baixa”. Municípios sem DAI ou abaixo do
mínimo aparecem como dados insuficientes, e não como categoria muito baixa.

A alternativa nutricional exige **DAI ≥6,7% E DPI ≥1,8%**, com pelo menos
20 avaliações **em cada relatório**. Mostram-se a comparação em todo o universo
válido para DAI e em um universo comum com ambos os denominadores válidos.
Assim é possível distinguir o efeito do critério adicional do efeito da falta
de informação de peso. A alternativa não muda a seleção principal.

In [9]:
resumo_oms = classificados["dai_categoria_oms"].value_counts().reindex(
    OMS_LABELS + ["dados insuficientes"], fill_value=0).rename_axis("dai_categoria_oms").reset_index(name="municipios")
display(tabela_br(resumo_oms))
comparacao_nutricional = nutrition_comparison(classificados)
display(tabela_br(comparacao_nutricional))
peso_insuficiente = classificados.loc[classificados["criterio_dpi"].isna(),
    ["codigo_ibge", "municipio", "uf", "avaliados_peso", "dpi_pct", "dpi_status"]]
display(tabela_br(peso_insuficiente))

,dai_categoria_oms,municipios
0,muito baixa,68
1,baixa,3.051
2,média,2.209
3,alta,208
4,muito alta,30
5,dados insuficientes,5


,universo,municipios,dai_isolado,dai_e_dpi,convergencia_dai,convergencia_dai_dpi
0,Todos os válidos para DAI,5.566,4.292,3.693,548,508
1,Denominadores válidos em altura e peso,5.566,4.292,3.693,548,508


,codigo_ibge,municipio,uf,avaliados_peso,dpi_pct,dpi_status
4320,4200754,ALTO BELA VISTA,SC,19,"0,00%",dados insuficientes
4814,4310462,IPIRANGA DO SUL,RS,10,"0,00%",dados insuficientes
4976,4317251,SANTA TEREZA,RS,6,"0,00%",dados insuficientes
5077,4322350,UNIAO DA SERRA,RS,15,"0,00%",dados insuficientes
5199,510183,BOA ESPERANCA DO NORTE,MT,0,—,dados insuficientes


## 8. Mapas nacionais

O mapa de **0/1/2/3 dimensões** mostra graus de convergência, não intensidade
clínica ou ranking. O mapa de **convergência total** destaca somente os
selecionados; dados insuficientes permanecem distintos dos não selecionados.
O mapa de **DAI por faixas OMS** descreve os acompanhamentos com denominador
mínimo. Mantêm-se os mapas numéricos e de disponibilidade úteis da primeira versão.

Polígonos maiores não representam mais pessoas. Proximidade visual não é
teste de autocorrelação espacial. Cinza indica ausência ou insuficiência,
nunca baixo risco. IDHM menor aparece mais vermelho; nos demais mapas
numéricos, valores maiores aparecem mais vermelhos.

In [10]:
print("Municípios da análise sem geometria:", int((~controle_geometria["tem_geometria"]).sum()))
print("Feições sem registro na análise:", len(universo_ibge - set(base["codigo_ibge_6"])))
figuras_mapas = []
if GERAR_MAPAS:
    mapa = classificados.copy()
    mapa["dimensoes_mapa"] = mapa["n_dimensoes_desfavoraveis"].astype("string").fillna("dados insuficientes")
    mapa["convergencia_mapa"] = mapa["convergencia_total"].map(
        {True: "Convergência total", False: "Sem convergência total"}).fillna("dados insuficientes")
    mapa["disponibilidade"] = mapa["elegivel_principal"].map(
        {True: "Elegível", False: "dados insuficientes"})
    mapas_categoricos = [
        ("dimensoes_mapa", "Número de dimensões desfavoráveis", "09_dimensoes", DIMENSION_COLORS),
        ("convergencia_mapa", "Convergência social, alimentar e nutricional", "11_convergencia_total",
            {"Convergência total": "#9e1b32", "Sem convergência total": "#edf2ef", "dados insuficientes": "#b9b9b9"}),
        ("dai_categoria_oms", "DAI — categorias de prevalência OMS", "12_dai_oms", OMS_COLORS),
        ("disponibilidade", "Disponibilidade para a classificação principal", "05_disponibilidade",
            {"Elegível": "#327c81", "dados insuficientes": "#b9b9b9"})]
    for coluna, titulo, nome, cores in mapas_categoricos:
        display(map_figure(geometria, mapa, coluna, titulo, PASTA_FIGURAS, nome, cores))
        figuras_mapas.append(nome)
        plt.close("all")
    for coluna, titulo, nome in [("ivs", "IVS — vulnerabilidade estrutural de 2010", "06_ivs"),
        ("idhm", "IDHM — 2010; menor é mais desfavorável", "10_idhm"),
        ("cadinsan_pct", f"CadInsan (%) — {CENARIO_CADINSAN}", "07_cadinsan"),
        ("dai_pct", f"DAI (%) — altura com n ≥ {MINIMO_AVALIADOS}", "08_dai")]:
        if coluna == "dai_pct":
            mapa[coluna] = mapa[coluna].where(mapa["avaliados_altura"].ge(MINIMO_AVALIADOS))
        display(map_figure(geometria, mapa, coluna, titulo, PASTA_FIGURAS, nome))
        figuras_mapas.append(nome)
        plt.close("all")

Municípios da análise sem geometria: 0
Feições sem registro na análise: 0


<Figure size 990x990 with 1 Axes>

<Figure size 990x990 with 1 Axes>

<Figure size 990x990 with 1 Axes>

<Figure size 990x990 with 1 Axes>

<Figure size 990x990 with 2 Axes>

<Figure size 990x990 with 2 Axes>

<Figure size 990x990 with 2 Axes>

<Figure size 990x990 with 2 Axes>

## 9. Distribuições, associações e correlações

Os histogramas mostram as distribuições, inclusive DAI/DPI em todos os
municípios com avaliações. Os diagramas relacionam os indicadores com DAI;
as linhas mostram cortes e o vermelho, convergência das três dimensões.

Spearman descreve associação monotônica, sem exigir distribuição normal.
Utilizam-se pares disponíveis e, para cada indicador nutricional, seu próprio
denominador mínimo. A matriz informa o número de municípios de cada par.
A relação inversa IVS–IDHM, se forte, reforça a decisão de agrupá-los na dimensão
social. Correlação ajuda a identificar redundância e associações, mas não
demonstra causalidade, relação individual nem independência das fontes.

In [11]:
display(distribution_figure(classificados, PASTA_FIGURAS))
plt.close("all")
display(association_figure(classificados, cortes, PASTA_FIGURAS))
plt.close("all")
figura_correlacao, correlacoes, n_pares = correlation_figure(classificados, PASTA_FIGURAS, MINIMO_AVALIADOS)
display(figura_correlacao)
plt.close("all")
rho_social = correlacoes.loc["ivs", "idhm"]
relacao_social = "forte e inversa" if rho_social <= -0.7 else "inversa" if rho_social < 0 else "não inversa"
display(Markdown(f"**IVS–IDHM:** Spearman = {numero_br(rho_social)}, "
    f"n = {contagem_br(n_pares.loc['ivs', 'idhm'])}; relação {relacao_social}. "
    "O agrupamento também tem justificativa conceitual; não depende apenas de um coeficiente."))

<Figure size 1540x770 with 6 Axes>

<Figure size 1650x495 with 3 Axes>

<Figure size 880x770 with 2 Axes>

**IVS–IDHM:** Spearman = -0,889, n = 5.565; relação forte e inversa. O agrupamento também tem justificativa conceitual; não depende apenas de um coeficiente.

## 10. Sensibilidade e estabilidade

Comparam-se **CadInsan P70/P75/P80/P90**, mínimos SISVAN **20/50/100** e
DAI **6,7% / 10% / P75**, nos cenários com/sem efeito do PBF.
**6,7%** é a referência do Mapa InSAN; **10%** inicia a categoria OMS média;
**P75** é uma comparação exploratória da distribuição municipal do DAI.
Somente o cenário P75 de DAI recalcula seu corte entre municípios acima
do mínimo de altura; os cortes fixos não variam com a distribuição.
O P75 alimentar usa todos os municípios com CadInsan válido, antes dos filtros.

As tabelas de uma mudança por vez mostram quem **entra/sai em relação ao
cenário principal**, não transições sucessivas entre linhas. O painel mostra
combinações no cenário CadInsan principal; a exportação contém todos os cenários.
IVS/IDHM permanecem fixos. Nenhuma alternativa substitui automaticamente
o resultado principal.

Jaccard compara as listas (interseção / união; 1 indica igualdade). Frequência
de seleção é fração das especificações testadas, não probabilidade de risco
nem intervalo de confiança. Os cenários têm sentidos diferentes e são identificados.

In [12]:
tabela_sensibilidade, estabilidade = sensitivity(
    base, classificados, sorted(set(QUANTIS_SENSIBILIDADE + [QUANTIL_CADINSAN])),
    sorted(set(MINIMOS_SENSIBILIDADE + [MINIMO_AVALIADOS])),
    sorted(set(CENARIOS_SENSIBILIDADE + [CENARIO_CADINSAN])),
    list(dict.fromkeys(CORTES_DAI_SENSIBILIDADE + [CORTE_DAI])))
principal_sens = tabela_sensibilidade["cenario"].eq(CENARIO_CADINSAN)
sensibilidade_cadinsan = tabela_sensibilidade.loc[principal_sens &
    tabela_sensibilidade["minimo_avaliados"].eq(MINIMO_AVALIADOS) &
    tabela_sensibilidade["referencia_dai"].eq(str(CORTE_DAI))]
sensibilidade_minimos = tabela_sensibilidade.loc[principal_sens &
    tabela_sensibilidade["quantil"].eq(QUANTIL_CADINSAN) &
    tabela_sensibilidade["referencia_dai"].eq(str(CORTE_DAI))]
sensibilidade_dai = tabela_sensibilidade.loc[principal_sens &
    tabela_sensibilidade["quantil"].eq(QUANTIL_CADINSAN) &
    tabela_sensibilidade["minimo_avaliados"].eq(MINIMO_AVALIADOS)]
for titulo, tabela in [("CadInsan — uma mudança por vez", sensibilidade_cadinsan),
                       ("Mínimo de avaliações — uma mudança por vez", sensibilidade_minimos),
                       ("DAI — uma mudança por vez", sensibilidade_dai)]:
    display(Markdown("### " + titulo))
    display(tabela_br(tabela))
display(sensitivity_figure(tabela_sensibilidade, PASTA_FIGURAS, CENARIO_CADINSAN))
plt.close("all")
classificados = classificados.merge(estabilidade, on="codigo_ibge_6", validate="one_to_one")
convergentes = classificados.loc[classificados["convergencia_total"].fillna(False)].sort_values(["uf", "municipio"])
display(tabela_br(convergentes[["municipio", "uf", "cenarios_elegiveis", "cenarios_selecionado",
    "cenarios_testados", "fracao_cenarios_selecionado"]].head(30)))

### CadInsan — uma mudança por vez

,cenario,quantil,minimo_avaliados,referencia_dai,corte_dai_pct,corte_cadinsan_pct,elegiveis,municipios_dim_alimentar,municipios_dim_nutricional,convergencia_total,entram_vs_principal,saem_vs_principal,coincidentes_principal,jaccard_principal
0,com_PBF,"0,70",20,6.7,"6,70%","10,16%",5.561,1.671,4.292,637,89,0,548,"0,860"
9,com_PBF,"0,75",20,6.7,"6,70%","10,97%",5.561,1.393,4.292,548,0,0,548,"1,000"
18,com_PBF,"0,80",20,6.7,"6,70%","11,92%",5.561,1.114,4.292,479,0,69,479,"0,874"
27,com_PBF,"0,90",20,6.7,"6,70%","14,97%",5.561,557,4.292,280,0,268,280,"0,511"


### Mínimo de avaliações — uma mudança por vez

,cenario,quantil,minimo_avaliados,referencia_dai,corte_dai_pct,corte_cadinsan_pct,elegiveis,municipios_dim_alimentar,municipios_dim_nutricional,convergencia_total,entram_vs_principal,saem_vs_principal,coincidentes_principal,jaccard_principal
9,com_PBF,"0,75",20,6.7,"6,70%","10,97%",5.561,1.393,4.292,548,0,0,548,"1,000"
12,com_PBF,"0,75",50,6.7,"6,70%","10,97%",5.531,1.393,4.278,548,0,0,548,"1,000"
15,com_PBF,"0,75",100,6.7,"6,70%","10,97%",5.326,1.393,4.163,547,0,1,547,"0,998"


### DAI — uma mudança por vez

,cenario,quantil,minimo_avaliados,referencia_dai,corte_dai_pct,corte_cadinsan_pct,elegiveis,municipios_dim_alimentar,municipios_dim_nutricional,convergencia_total,entram_vs_principal,saem_vs_principal,coincidentes_principal,jaccard_principal
9,com_PBF,"0,75",20,6.7,"6,70%","10,97%",5.561,1.393,4.292,548,0,0,548,"1,000"
10,com_PBF,"0,75",20,10.0,"10,00%","10,97%",5.561,1.393,2.447,390,0,158,390,"0,712"
11,com_PBF,"0,75",20,P75,"12,37%","10,97%",5.561,1.393,1.392,244,0,304,244,"0,445"


<Figure size 1650x495 with 3 Axes>

,municipio,uf,cenarios_elegiveis,cenarios_selecionado,cenarios_testados,fracao_cenarios_selecionado
53,ASSIS BRASIL,AC,72,63,72,"0,875"
56,CAPIXABA,AC,72,42,72,"0,583"
62,MANOEL URBANO,AC,72,63,72,"0,875"
63,MARECHAL THAUMATURGO,AC,72,54,72,"0,750"
73,PORTO ACRE,AC,72,24,72,"0,333"
67,RODRIGUES ALVES,AC,72,63,72,"0,875"
68,SANTA ROSA DO PURUS,AC,72,27,72,"0,375"
1654,BARRA DE SANTO ANTONIO,AL,72,18,72,"0,250"
1668,CARNEIROS,AL,72,45,72,"0,625"
1669,CHA PRETA,AL,72,36,72,"0,500"


## 11. Limitações metodológicas

- **Temporalidade:** IVS/IDHM de 2010 descrevem vulnerabilidade estrutural
  histórica; não são uma fotografia contemporânea. CadInsan é de janeiro/2025
  e SISVAN, de 2025. As fontes não medem a mesma população no mesmo instante.
- **SISVAN:** os acompanhados pelos serviços de saúde não constituem
  necessariamente amostra probabilística de todos os residentes. DAI/DPI
  descrevem esse público, não estimativas municipais representativas garantidas.
- **Denominadores:** ≥20 é precedente operacional, não garantia de estabilidade.
  Os mínimos são separados em altura/peso; não há ligação individual entre eles.
- **CadInsan:** risco estimado no universo familiar analisado; P75 é exploratório,
  não ponto de corte oficial. Percentis municipais não têm ponderação populacional.
- **Referências nutricionais:** limiares do Mapa InSAN são adaptados a outro
  público SISVAN. Categorias OMS auxiliam interpretação, sem corrigir viés de cobertura.
- **Território e associação:** verificar códigos não harmoniza limites históricos.
  A malha é ilustrativa. Sobreposição e correlação não demonstram causalidade
  nem permitem inferir situações individuais.
- **Classificação:** dados insuficientes não significam baixo risco, e zero
  dimensões não demonstra ausência de insegurança alimentar. As classes não
  substituem diagnóstico local, escuta comunitária ou levantamentos representativos.

## 12. Conclusões descritivas

A síntese abaixo é produzida a partir dos resultados desta execução.
A convergência pode orientar investigação territorial e discussão de políticas,
sem estabelecer ranking nacional, escore ponderado ou classificação oficial.

In [13]:
selecionados = len(convergentes)
regiao_maior_numero = resumo_regional.sort_values("convergencia_total", ascending=False).iloc[0]
resumo_execucao = f"""# Síntese da análise municipal — v2

Na união das fontes há **{contagem_br(len(base))} municípios**;
**{contagem_br(elegiveis)}** têm informação suficiente nas três dimensões e
**{contagem_br(selecionados)}** apresentam convergência total.
Os selecionados presentes na malha representam
**{percentual_br(selecionados_na_malha / len(universo_ibge) * 100)}** de seus
{contagem_br(len(universo_ibge))} municípios de referência, e a seleção corresponde a
**{percentual_br(selecionados / elegiveis * 100 if elegiveis else np.nan)}** dos elegíveis.

A maior quantidade selecionada está em **{regiao_maior_numero['regiao']}**
({contagem_br(regiao_maior_numero['convergencia_total'])} municípios);
quantidade não equivale à maior proporção nem ao maior risco individual.

Regra principal: social (IVS ≥0,401 E IDHM <0,600), alimentar
(CadInsan ≥P{QUANTIL_CADINSAN * 100:.0f}, cenário {CENARIO_CADINSAN}) e nutricional
(DAI ≥{numero_br(CORTE_DAI, 1)}%, altura com n ≥{MINIMO_AVALIADOS}).
DPI é complementar. A alternativa social+alimentar+DAI+DPI seleciona
{contagem_br(int(classificados['convergencia_total_dai_dpi'].sum()))} municípios.

A sensibilidade completa variou de
{contagem_br(tabela_sensibilidade['convergencia_total'].min())} a
{contagem_br(tabela_sensibilidade['convergencia_total'].max())} municípios em
{len(tabela_sensibilidade)} especificações. Essa variação não é intervalo de confiança;
consultar as mudanças de uma escolha por vez antes de avaliar estabilidade.

IVS–IDHM: Spearman {numero_br(rho_social)}, associação {relacao_social},
coerente com seu tratamento conjunto na dimensão social quando inversa.
IVS/IDHM são históricos (2010); CadInsan e SISVAN, de 2025.
Os resultados descrevem convergência nos universos analisados, não a
prevalência de fome em toda a população e não permitem inferir causalidade.
"""
display(Markdown(resumo_execucao))

# Síntese da análise municipal — v2

Na união das fontes há **5.571 municípios**;
**5.561** têm informação suficiente nas três dimensões e
**548** apresentam convergência total.
Os selecionados presentes na malha representam
**9,84%** de seus
5.571 municípios de referência, e a seleção corresponde a
**9,85%** dos elegíveis.

A maior quantidade selecionada está em **Nordeste**
(406 municípios);
quantidade não equivale à maior proporção nem ao maior risco individual.

Regra principal: social (IVS ≥0,401 E IDHM <0,600), alimentar
(CadInsan ≥P75, cenário com_PBF) e nutricional
(DAI ≥6,7%, altura com n ≥20).
DPI é complementar. A alternativa social+alimentar+DAI+DPI seleciona
508 municípios.

A sensibilidade completa variou de
137 a
672 municípios em
72 especificações. Essa variação não é intervalo de confiança;
consultar as mudanças de uma escolha por vez antes de avaliar estabilidade.

IVS–IDHM: Spearman -0,889, associação forte e inversa,
coerente com seu tratamento conjunto na dimensão social quando inversa.
IVS/IDHM são históricos (2010); CadInsan e SISVAN, de 2025.
Os resultados descrevem convergência nos universos analisados, não a
prevalência de fome em toda a população e não permitem inferir causalidade.


## 13. Exportação e reprodução

Exporta-se a base municipal completa (incluindo dados insuficientes), a lista
de convergência, flags anuláveis, tabelas de cobertura/qualidade, categorias OMS,
correlações com tamanhos dos pares, sensibilidade, auditorias e figuras PNG/SVG.
O CSV final inclui códigos, território, indicadores, denominadores, três dimensões
e suas classes. A auditoria de arredondamento refere-se apenas ao CadInsan.

O ZIP contém dicionário, síntese e manifesto com parâmetros, fórmulas, hashes
das entradas e do código, procedência e versões do ambiente. Não contém a
base adicional de pessoas cadastradas nem colunas derivadas dela.
Pode ser baixado pela aba de arquivos do Colab ou pela opção nos parâmetros.

In [14]:
identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = PASTA_SAIDA / f"execucao_{identificador}"
destino.mkdir()
tabelas = {"base_municipal_final": classificados, "municipios_convergencia_total": convergentes,
    "cortes": cortes, "controle_integracao": controle_integracao, "municipios_sem_correspondencia": ausencias,
    "sem_classificacao": sem_classificacao, "qualidade_sisvan": qualidade_sisvan,
    "interpretacao_sisvan": interpretacao_sisvan, "controle_geometria": controle_geometria,
    "descritivas": descritivas.rename_axis("indicador").reset_index(), "totais_nutricionais": totais_nutricionais,
    "resumo_nacional": resumo_nacional, "resumo_uf": resumo_uf, "resumo_regional": resumo_regional,
    "resumo_classes": resumo_classes, "resumo_social": resumo_social, "resumo_oms": resumo_oms,
    "comparacao_nutricional": comparacao_nutricional, "peso_insuficiente": peso_insuficiente,
    "sensibilidade": tabela_sensibilidade, "sensibilidade_cadinsan": sensibilidade_cadinsan,
    "sensibilidade_minimos": sensibilidade_minimos, "sensibilidade_dai": sensibilidade_dai,
    "estabilidade": estabilidade, "dicionario_variaveis": dictionary(), "validacao_catalogo": validacao_catalogo,
    "comparacao_arredondamento": comparacao_arredondamento, "cortes_percentuais_csv": cortes_percentuais_csv,
    "correlacoes_spearman": correlacoes.rename_axis("indicador").reset_index(),
    "n_pares_correlacao": n_pares.rename_axis("indicador").reset_index()}
for nome, tabela in tabelas.items():
    tabela.to_csv(destino / f"{nome}.csv", index=False, encoding="utf-8-sig")
(destino / "figuras").mkdir()
for nome in ["01_distribuicoes", "02_associacoes", "03_correlacoes", "04_sensibilidade"] + figuras_mapas:
    for extensao in ["png", "svg"]:
        shutil.copy2(PASTA_FIGURAS / f"{nome}.{extensao}", destino / "figuras")
manifesto = {
    "versao": 2, "executado_em": datetime.now(timezone.utc).isoformat(),
    "quantil_cadinsan": QUANTIL_CADINSAN, "corte_dai_pct": CORTE_DAI,
    "corte_dpi_pct_complementar": CORTE_DPI, "minimo_avaliados": MINIMO_AVALIADOS,
    "cenario_cadinsan": CENARIO_CADINSAN,
    "formula_cadinsan_pct": "100 * familias_risco_cenario / familias_cadinsan",
    "formula_dai_pct": "100 * (altura_muito_baixa_n + altura_baixa_n) / avaliados_altura",
    "formula_dpi_pct": "100 * (peso_muito_baixo_n + peso_baixo_n) / avaliados_peso",
    "dimensoes": {"social": "IVS >=0.401 E IDHM <0.600", "alimentar": "CadInsan >= quantil nacional",
                  "nutricional": "DAI >= corte principal E altura com denominador mínimo"},
    "regra": "Convergência das três dimensões; DPI não integra seleção principal",
    "indicadores_com_quantil_principal": ["cadinsan"], "cortes": cortes.to_dict("records"),
    "hashes_entrada": hashes_entrada.to_dict("records"), "codigo_analise_sha256": CODIGO_ANALISE_SHA256,
    "proveniencia_catalogo": proveniencia_catalogo,
    "proveniencia_sisvan": {s: sisvan_metadata(PASTA_DADOS, s) for s in ["sisvan_altura", "sisvan_peso"]},
    "malha": metadados_malha, "universo_nacional": resumo_nacional.to_dict("records"),
    "ambiente": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                 "matplotlib": matplotlib.__version__, "scipy": scipy.__version__},
    "sensibilidade": tabela_sensibilidade.to_dict("records"),
    "limites": ["Temporalidade 2010/2025", "Público SISVAN não necessariamente representativo",
                "Adaptação das referências nutricionais; não reprodução do Mapa InSAN",
                "Corte CadInsan exploratório", "Sem cobertura populacional ou inferência causal"]}
(destino / "manifesto_execucao.json").write_text(json.dumps(manifesto, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(destino / "resumo.md").write_text(resumo_execucao, encoding="utf-8")
shutil.copytree(PASTA_DADOS / "documentacao", destino / "documentacao_entradas")
shutil.copy2(PASTA_DADOS / "README.md", destino / "documentacao_entradas/README_SELECAO.md")
shutil.copy2(PASTA_DADOS / "SHA256SUMS", destino / "documentacao_entradas")
arquivo_zip = shutil.make_archive(str(destino), "zip", root_dir=destino)
print("ZIP pronto:", arquivo_zip)
if BAIXAR_RESULTADOS_NO_COLAB:
    try:
        from google.colab import files
    except ImportError:
        print("Fora do Colab: utilizar o ZIP no caminho informado.")
    else:
        files.download(arquivo_zip)

ZIP pronto: /home/reddyyz/projects/ic/areas-risco-alimentar-e-vuln-social/resultados_sobreposicao_v2/resultados/execucao_20261005T181716220850Z.zip


## Referências metodológicas

- Ipea (2015). [Atlas da Vulnerabilidade Social nos Municípios Brasileiros](https://repositorio.ipea.gov.br/bitstream/11058/4381/1/Atlas_da_vulnerabilidade_social_nos_municipios_brasileiros.pdf), conceito, subíndices e faixas de IVS.
- PNUD, Ipea e FJP. [Atlas do Desenvolvimento Humano no Brasil](https://www.undp.org/pt/brazil/desenvolvimento-humano/atlas-do-desenvolvimento-humano-no-brasil); [faixas de IDHM, anexo estatístico](https://www.undp.org/sites/g/files/zskgke326/files/2024-05/anexo_estatistico_pnud_21maio24_isbn_web2.pdf).
- MDS. [CadInsan 2025](https://www.gov.br/mds/pt-br/Sisan/vigilancia-do-sisan/CADINSAN2025.pdf), metodologia e universo (p. 7), cenários (p. 16–19) e tabela municipal (a partir de p. 22).
- Ministério da Saúde. [SISVAN](https://www.gov.br/saude/pt-br/composicao/saps/vigilancia-alimentar-e-nutricional/sisvan) e [relatórios públicos](https://sisaps.saude.gov.br/sisvan/relatoriopublico/).
- MDS. [Mapa InSAN 2017–2022](https://www.gov.br/mds/pt-br/Sisan/monitoramento-da-san/MapaInSAN_20172022.pdf), seção de mapeamento/tabela 2: ≥20 acompanhamentos, DAI ≥6,7%, DPI ≥1,8%. Esses valores foram confirmados no texto oficial indexado; o acesso direto ao PDF apresentou restrição durante a revisão.
- Ministério da Saúde (2009). [PNDS 2006 — Dimensões do Processo Reprodutivo e da Saúde da Criança](https://bvsms.saude.gov.br/bvs/publicacoes/pnds_crianca_mulher.pdf), p. 217, referência citada pelo Mapa InSAN.
- OMS. [NLiS — Malnutrition in children](https://www.who.int/data/nutrition/nlis/info/malnutrition-in-children), tabela de prevalências de stunting; de Onis et al. (2018), DOI [10.1017/S1368980018002434](https://doi.org/10.1017/S1368980018002434).
- [Cozinhas Solidárias](https://github.com/TriangulosTecnologia/cozsolidarias): procedência dos dois CSVs sociais utilizados; catálogo correspondente, com verificação de hashes.
- IBGE. [API de malhas simplificadas v4](https://servicodados.ibge.gov.br/api/docs/malhas?versao=4).